# DL4CV PA02-Q4: Batch Normalization


---

### $\color{red}{\textbf{Name and ID:}}$ XXXXXXXX YourName(Fill here)

Before we start, please put your student ID and your name in following format.
For example

12345678 HONGGildong


---

### Submission

1. The output of the code cells in this `.ipynb` file $\color{red}{\text{MUST}}$ be saved.
   - Save text output of code cells
   - Save graphs and plots of code cells    
2. All The `.py` files should be submitted.
3. zip the required `.ipynb` and `.py` files into $\color{red}{\mathtt{StudentID\_StudentName.zip}}$ and submit.
   - For example `12345678_HONGGildong.zip`
4. Deadline is $\color{red}{2026:10:17-23:59:59}$ (YYYY:MM:DD-HH:MM:SS)

---

### To-do

1. Fill in Your ID and Name.
2. (Programming) In `layers.py`, implement **batchnorm_forward()** function.
3. (Programming) In `layers.py`, implement **batchnorm_backward()** function.
4. (Programming) In `layers.py`, implement **layernorm_forward()** function.
5. (Programming) In `layers.py`, implement **layernorm_back()** function.
6. (Programming) In `two_layer_net.py`, implement **`__init__()`** function in `class TwoLayerNet` with `'batchnorm'` and `'layernorm'`.
7. (Programming) In `two_layer_net.py`, implement **loss()** function in `class TwoLayerNet` with `'batchnorm'` and `'layernorm'`.
8.  (Save Outputs) Successfully Run all the **code cells** in this `.ipynb`, $\color{red}{\text{MUST}}$ save the outputs.
9.  Submit $\color{red}{\mathtt{StudentID\_StudentName.zip}}$ file to Plato before deadline.


---

## Batch Normalization Programming Assignment

In this programming assignment, we implement **Batch Normalization (BN)** and **Layer Normalization (LN)** using their mathematical forward and backward passes. The goal is to understand how normalization layers transform activations during the forward pass and how gradients propagate through them during backpropagation.

In this Programming Assignment you will:
    
- implement forward and backward pass of batch normalization layer.
- implement forward and backward pass of layer normalization layer.
- implement two-layer-net with `batch normalization` and `layer normalization`.

---

## Setup Code

### Google Colab Setup (Skip If Local Machine)

Unzip the `PA02.zip` file, upload entire `DeepLearning4ComputerVision` folder into Google drivers `My Drive` folder.

Next we need to run a few commands to set up our environment on Google Colab. If you are running this notebook on a local machine you can skip this section.

Run the following cell to mount your Google Drive. Follow the link, sign in to your Google account (the same account you used to store this notebook!) and copy the authorization code into the text box that appears below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'DeepLearning4ComputerVision/PA02'

import os
import sys

#sys.path.append('/content/drive/My Drive/{}'.format(GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)) # if not working, try this line instead
os.chdir('/content/drive/My Drive/{}'.format(GOOGLE_DRIVE_PATH_AFTER_MYDRIVE))

# Print current working directory
print("Current Working Directory:", os.getcwd())

# List files to make sure your data is visible
print("Available Files:", os.listdir())

If the output shows
> Mounted at /content/drive
> 
> Current Working Directory: /content/drive/My Drive/DeepLearning4ComputerVision/PA02
> 
> Available Files: ['datasets', 'dlcv', 'knn.py', 'softmax.py', ...]

Then, the Google Colab is working correctly.

### python env and ipynb setup (Local Machine Run from here) (Google Colab run this)

In [ ]:
import sys
import importlib
sys.modules["imp"] = importlib
# Notebook will reload external python modules;
%load_ext autoreload
%autoreload 2

import numpy as np

import matplotlib.pyplot as plt
# This is a bit of magic to make matplotlib figures appear inline in the notebook
# rather than in a new window.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'


def rel_error(x, y):
  """ returns relative error """
  return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))



def print_mean_std(x,axis=0):
  print(f"  means: {x.mean(axis=axis)}")
  print(f"  stds:  {x.std(axis=axis)}\n")

## CIFAR-10 Data Loading and Preprocessing

In [ ]:
from dlcv.data_utils import get_CIFAR10_data
# Load the (preprocessed) CIFAR10 data.
cifar10_dir = 'datasets/cifar-10-batches-py'

data = get_CIFAR10_data(cifar10_dir)
num_train = 1000
small_data = {
  'X_train': data['X_train'][:num_train],
  'y_train': data['y_train'][:num_train],
  'X_val': data['X_val'],
  'y_val': data['y_val'],
}
for k, v in list(small_data.items()):
  print(('%s: ' % k, v.shape))

## Batch Normalization

### Implement batch normalization forward pass (train)

In `layers.py`, implement **batchnorm_forward()** function. Implement the `'train'` forward pass.

In [ ]:
# Check the training-time forward pass by checking means and variances
# of features both before and after batch normalization   
from layers import batchnorm_forward
# Simulate the forward pass for a two-layer network.
np.random.seed(231)
N, D1, D2, D3 = 200, 50, 60, 3
X = np.random.randn(N, D1)
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)
a = np.maximum(0, X.dot(W1)).dot(W2)

print('Before batch normalization:')
print_mean_std(a,axis=0)

gamma = np.ones((D3,))
beta = np.zeros((D3,))

# Means should be close to zero and stds close to one.
print('After batch normalization (gamma=1, beta=0)')
a_norm, _ = batchnorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=0)

gamma = np.asarray([1.0, 2.0, 3.0])
beta = np.asarray([11.0, 12.0, 13.0])

# Now means should be close to beta and stds close to gamma.
print('After batch normalization (gamma=', gamma, ', beta=', beta, ')')
a_norm, _ = batchnorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=0)

### Implement batch normalization forward pass (test)

In `layers.py`, implement **batchnorm_forward()** function. Implement the `'test'` forward pass.

In [ ]:
# Check the test-time forward pass by running the training-time
# forward pass many times to warm up the running averages, and then
# checking the means and variances of activations after a test-time
# forward pass.

np.random.seed(231)
N, D1, D2, D3 = 200, 50, 60, 3
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)

bn_param = {'mode': 'train'}
gamma = np.ones(D3)
beta = np.zeros(D3)

for t in range(50):
  X = np.random.randn(N, D1)
  a = np.maximum(0, X.dot(W1)).dot(W2)
  batchnorm_forward(a, gamma, beta, bn_param)

bn_param['mode'] = 'test'
X = np.random.randn(N, D1)
a = np.maximum(0, X.dot(W1)).dot(W2)
a_norm, _ = batchnorm_forward(a, gamma, beta, bn_param)

# Means should be close to zero and stds close to one, but will be
# noisier than training-time forward passes.
print('After batch normalization (test-time):')
print_mean_std(a_norm,axis=0)

### Implement batch normalization backward pass

In `layers.py`, implement **batchnorm_backward()** function. 

In [ ]:
from dlcv.gradient_check import eval_numerical_gradient_array
from layers import batchnorm_backward
# Gradient check batchnorm backward pass.
np.random.seed(231)
N, D = 4, 5
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)

bn_param = {'mode': 'train'}
fx = lambda x: batchnorm_forward(x, gamma, beta, bn_param)[0]
fg = lambda a: batchnorm_forward(x, a, beta, bn_param)[0]
fb = lambda b: batchnorm_forward(x, gamma, b, bn_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma.copy(), dout)
db_num = eval_numerical_gradient_array(fb, beta.copy(), dout)

_, cache = batchnorm_forward(x, gamma, beta, bn_param)
dx, dgamma, dbeta = batchnorm_backward(dout, cache)

# You should expect to see relative errors between 1e-13 and 1e-8.
print('dx error: ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(da_num, dgamma))
print('dbeta error: ', rel_error(db_num, dbeta))

### Implement Two-layer-Net, apply batch normalization.

In `two_layer_net.py`, implement **`__init__()`** function in `class TwoLayerNet` with `'batchnorm'`.

In `two_layer_net.py`, implement **loss()** function in `class TwoLayerNet` with `'batchnorm'`.

For the Batch Normalization Two-Layer-Net model:

$$
\boxed{
X
\rightarrow
\text{Linear}_1
\rightarrow
\text{BatchNorm}
\rightarrow
\text{ReLU}
\rightarrow
\text{Linear}_2
\rightarrow
\text{Softmax}
\rightarrow
\text{Cross-Entropy}.
}
$$

In [ ]:
from dlcv.gradient_check import eval_numerical_gradient
from two_layer_net import TwoLayerNet
np.random.seed(231)
N, input_dim, hidden_size, num_classes = 2, 15, 20, 10
X = np.random.randn(N, input_dim)
y = np.random.randint(num_classes, size=(N,))

# You should expect losses between 1e-4~1e-10 for W, 
# losses between 1e-08~1e-10 for b,
# and losses between 1e-08~1e-09 for beta and gammas.
model = TwoLayerNet(input_dim, hidden_size, num_classes, reg=0.01, normalization='batchnorm')


loss, grads = model.loss(X, y)
print('Initial loss: ', loss)

for name in sorted(grads):
  f = lambda _: model.loss(X, y)[0]
  grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
  print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))


### Train Two-layer-net model with Cifar10 data

In [ ]:
from dlcv.solver import Solver
np.random.seed(231)

input_size = 32 * 32 * 3
hidden_size = 200
num_classes = 10

bn_model = TwoLayerNet(input_size, hidden_size, num_classes, normalization='batchnorm')
model = TwoLayerNet(input_size, hidden_size, num_classes, normalization=None)

print('Solver with batch norm:')
bn_solver = Solver(bn_model, small_data,
                num_epochs=10, batch_size=50,
                update_rule='adam',
                optim_config={
                  'learning_rate': 1e-3,
                },
                verbose=True,print_every=20)
bn_solver.train()

print('\nSolver without batch norm:')
solver = Solver(model, small_data,
                num_epochs=10, batch_size=50,
                update_rule='adam',
                optim_config={
                  'learning_rate': 1e-3,
                },
                verbose=True, print_every=20)
solver.train()

In [ ]:
def plot_training_history(title, label, baseline, bn_solvers, plot_fn, bl_marker='.', bn_marker='.', labels=None):
    """utility function for plotting training history"""
    plt.title(title)
    plt.xlabel(label)
    bn_plots = [plot_fn(bn_solver) for bn_solver in bn_solvers]
    bl_plot = plot_fn(baseline)
    num_bn = len(bn_plots)
    for i in range(num_bn):
        label='with_norm'
        if labels is not None:
            label += str(labels[i])
        plt.plot(bn_plots[i], bn_marker, label=label)
    label='baseline'
    if labels is not None:
        label += str(labels[0])
    plt.plot(bl_plot, bl_marker, label=label)
    plt.legend(loc='lower center', ncol=num_bn+1) 

    
plt.subplot(3, 1, 1)
plot_training_history('Training loss','Iteration', solver, [bn_solver], \
                      lambda x: x.loss_history, bl_marker='o', bn_marker='o')
plt.subplot(3, 1, 2)
plot_training_history('Training accuracy','Epoch', solver, [bn_solver], \
                      lambda x: x.train_acc_history, bl_marker='-o', bn_marker='-o')
plt.subplot(3, 1, 3)
plot_training_history('Validation accuracy','Epoch', solver, [bn_solver], \
                      lambda x: x.val_acc_history, bl_marker='-o', bn_marker='-o')

plt.gcf().set_size_inches(15, 15)
plt.show()

## Layer Normalization

### Implement layer normalization forward pass

In `layers.py`, implement **layernorm_forward()** function.

In [ ]:
from layers import layernorm_forward
# Check the training-time forward pass by checking means and variances
# of features both before and after layer normalization.

# Simulate the forward pass for a two-layer network.
np.random.seed(231)
N, D1, D2, D3 =4, 50, 60, 3
X = np.random.randn(N, D1)
W1 = np.random.randn(D1, D2)
W2 = np.random.randn(D2, D3)
a = np.maximum(0, X.dot(W1)).dot(W2)

print('Before layer normalization:')
print_mean_std(a,axis=1)

gamma = np.ones(D3)
beta = np.zeros(D3)

# Means should be close to zero and stds close to one.
print('After layer normalization (gamma=1, beta=0)')
a_norm, _ = layernorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=1)

gamma = np.asarray([3.0,3.0,3.0])
beta = np.asarray([5.0,5.0,5.0])

# Now means should be close to beta and stds close to gamma.
print('After layer normalization (gamma=', gamma, ', beta=', beta, ')')
a_norm, _ = layernorm_forward(a, gamma, beta, {'mode': 'train'})
print_mean_std(a_norm,axis=1)

### Implement layer normalization backward pass

In `layers.py`, implement **layernorm_forward()** function.

In [ ]:
from layers import layernorm_backward
# Gradient check batchnorm backward pass.
np.random.seed(231)
N, D = 4, 5
x = 5 * np.random.randn(N, D) + 12
gamma = np.random.randn(D)
beta = np.random.randn(D)
dout = np.random.randn(N, D)

ln_param = {}
fx = lambda x: layernorm_forward(x, gamma, beta, ln_param)[0]
fg = lambda a: layernorm_forward(x, a, beta, ln_param)[0]
fb = lambda b: layernorm_forward(x, gamma, b, ln_param)[0]

dx_num = eval_numerical_gradient_array(fx, x, dout)
da_num = eval_numerical_gradient_array(fg, gamma.copy(), dout)
db_num = eval_numerical_gradient_array(fb, beta.copy(), dout)

_, cache = layernorm_forward(x, gamma, beta, ln_param)
dx, dgamma, dbeta = layernorm_backward(dout, cache)

# You should expect to see relative errors between 1e-12 and 1e-8.
print('dx error: ', rel_error(dx_num, dx))
print('dgamma error: ', rel_error(da_num, dgamma))
print('dbeta error: ', rel_error(db_num, dbeta))

### Implement Two-layer-Net, apply layer normalization.

In `two_layer_net.py`, implement **`__init__()`** function in `class TwoLayerNet` with `'layernorm'`.

In `two_layer_net.py`, implement **loss()** function in `class TwoLayerNet` with `'layernorm'`.


For the Layer Normalization Two-Layer-Net model:

$$
\boxed{
X
\rightarrow
\text{Linear}_1
\rightarrow
\text{LayerNorm}
\rightarrow
\text{ReLU}
\rightarrow
\text{Linear}_2
\rightarrow
\text{Softmax}
\rightarrow
\text{Cross-Entropy}.
}
$$

In [ ]:
from dlcv.gradient_check import eval_numerical_gradient
from two_layer_net import TwoLayerNet
np.random.seed(231)
N, input_dim, hidden_size, num_classes = 2, 15, 20, 10
X = np.random.randn(N, input_dim)
y = np.random.randint(num_classes, size=(N,))

# You should expect losses between 1e-4~1e-10 for W, 
# losses between 1e-08~1e-10 for b,
# and losses between 1e-08~1e-09 for beta and gammas.
model = TwoLayerNet(input_dim, hidden_size, num_classes, reg=0.01, normalization='layernorm')

loss, grads = model.loss(X, y)
print('Initial loss: ', loss)

for name in sorted(grads):
  f = lambda _: model.loss(X, y)[0]
  grad_num = eval_numerical_gradient(f, model.params[name], verbose=False, h=1e-5)
  print('%s relative error: %.2e' % (name, rel_error(grad_num, grads[name])))


In [ ]:
from dlcv.solver import Solver
np.random.seed(231)

input_size = 32 * 32 * 3
hidden_size = 200
num_classes = 10

weight_scale = 2e-2
ln_model = TwoLayerNet(input_size, hidden_size, num_classes, normalization='layernorm')

print('Solver with layer norm:')
ln_solver = Solver(ln_model, small_data,
                num_epochs=10, batch_size=50,
                update_rule='adam',
                optim_config={ 
                  'learning_rate': 1e-3,
                },
                verbose=True,print_every=20)
ln_solver.train()

In [ ]:
def plot_training_history(title, label, baseline, bn_solvers, plot_fn, bl_marker='.', bn_marker='.', labels=None):
    """utility function for plotting training history"""
    plt.title(title)
    plt.xlabel(label)
    bn_plots = [plot_fn(bn_solver) for bn_solver in bn_solvers]
    bl_plot = plot_fn(baseline)
    num_bn = len(bn_plots)
    labels = ['batchnorm', 'layernorm']
    for i in range(num_bn):
        label=labels[i]        
        plt.plot(bn_plots[i], bn_marker, label=label)
    label='baseline'
    
    plt.plot(bl_plot, bl_marker, label=label)
    plt.legend(loc='lower center', ncol=num_bn+1) 

    
plt.subplot(3, 1, 1)
plot_training_history('Training loss','Iteration', solver, [bn_solver, ln_solver], \
                      lambda x: x.loss_history, bl_marker='o', bn_marker='o')
plt.subplot(3, 1, 2)
plot_training_history('Training accuracy','Epoch', solver, [bn_solver, ln_solver], \
                      lambda x: x.train_acc_history, bl_marker='-o', bn_marker='-o')
plt.subplot(3, 1, 3)
plot_training_history('Validation accuracy','Epoch', solver, [bn_solver, ln_solver], \
                      lambda x: x.val_acc_history, bl_marker='-o', bn_marker='-o')

plt.gcf().set_size_inches(15, 15)
plt.show()